In [ ]:
"""
Objective: Perform unit root tests (ADF), estimate a Vector Autoregressive (VAR) model, 
and compute Forecast Error Variance Decomposition (FEVD) to analyze spillover risks 
between credit segments and the macroeconomy.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. DATA LOADING AND TRANSFORMATION
# ==========================================
def load_and_transform(csv_file: str) -> pd.DataFrame:
    """Loads the dataset, filters the extended window, and adds real GDP growth."""
    df = pd.read_csv(csv_file, index_col=0, parse_dates=True)
    df_model = df.loc['2000-01-01':'2026-04-01'].copy()
    
    # Real GDP is transformed into a quarterly percentage change
    df_model['gdp_growth'] = df_model['real_gdp'].pct_change() * 100
    return df_model.dropna()

# ==========================================
# 2. STATIONARITY TEST (ADF)
# ==========================================
def enforce_stationarity(df: pd.DataFrame, columns: list, p_limit: float = 0.05) -> pd.DataFrame:
    """Runs ADF tests for each series and applies first differencing if necessary."""
    stationary_df = pd.DataFrame(index=df.index)
    
    print("[*] Checking stationarity (ADF)...")
    for col in columns:
        p_value = adfuller(df[col])[1]
        if p_value > p_limit:
            stationary_df[col] = df[col].diff()
        else:
            stationary_df[col] = df[col]
            
    return stationary_df.dropna()

# ==========================================
# 3. SPILLOVER MATRIX & VISUALIZATION
# ==========================================
def plot_spillover_matrix(spillover_df: pd.DataFrame, output_file: str):
    """Generates the Connectivity Heatmap and exports it as an image."""
    label_names = [
        'Unemployment', 'GDP\nGrowth', 'Fed\nFunds', 'Spread\n(10Y-3M)', 
        'Delinquency\n(Mortgages)', 'Delinquency\n(Consumer)', 'Delinquency\n(Credit Cards)'
    ]
    spillover_df.index = label_names
    spillover_df.columns = label_names

    plt.figure(figsize=(11, 9))
    sns.heatmap(spillover_df, annot=True, cmap='Reds', fmt='.1f', 
                cbar_kws={'label': '% of Explained Variance'})

    plt.ylabel('Affected Variable (Receives Impact)', fontweight='bold')
    plt.xlabel('Shock Origin (Generates Impact)', fontweight='bold')
    plt.xticks(rotation=0, fontsize=8.5)
    plt.yticks(rotation=0, fontsize=10)
    plt.tight_layout()
    plt.savefig(output_file, dpi=300)
    print(f"\n[*] Connectivity chart exported: '{output_file}'.")

if __name__ == "__main__":
    # Macrofinancial system variables
    var_cols = [
        'unemployment', 'gdp_growth', 'fed_funds', 'spread_10y3m', 
        'delinq_mortgages', 'delinq_consumer', 'delinq_credit_cards'
    ]
    
    # 1. Dataset prep
    raw_df = load_and_transform('clean_macro_fred_dataset.csv')
    svar_df = raw_df[var_cols].dropna()
    stationary_df = enforce_stationarity(svar_df, var_cols)
    
    # 2. Lag selection (BIC) and VAR estimation
    print("[*] Optimizing VAR order (BIC)...")
    model = VAR(stationary_df)
    optimal_p = model.select_order(maxlags=4).bic
    results = model.fit(optimal_p)
    
    # 3. Forecast Error Variance Decomposition (FEVD) over 8 quarters
    horizon = 8
    fevd = results.fevd(horizon)
    
    spillover_matrix = pd.DataFrame(
        np.zeros((len(var_cols), len(var_cols))),
        index=var_cols, columns=var_cols
    )
    for i, col in enumerate(var_cols):
        spillover_matrix.iloc[i, :] = fevd.decomp[i][-1, :] * 100
        
    print("\n--- Predictive Ranking (Impact on Unemployment) ---")
    unemployment_impact = spillover_matrix.loc['unemployment', ['delinq_mortgages', 'delinq_consumer', 'delinq_credit_cards']]
    print(unemployment_impact.sort_values(ascending=False).to_string(float_format="{:.2f}%".format))
    
    # 4. Final plot
    plot_spillover_matrix(spillover_matrix, '01_spillover_matrix.png')